#### **What are Tests in dbt?**

In **dbt**, **tests are automated data quality checks** that validate whether your data matches your expectations.

Think of dbt tests as **guardrails for your data warehouse 🚧**

They answer questions like:

- ❌ Are there **nulls** where there shouldn’t be?

- ❌ Are there **duplicate IDs**?

- ❌ Are foreign keys **broken**?

- ❌ Does the data satisfy **business rules**?

If a test **fails**, dbt tells you **exactly what broke and where.**


---------------------


#### **1️⃣ Why do we need tests in dbt?**

In analytics engineering, **bad data = wrong decisions.**

Common real-world problems:

- Orders without customers

- Duplicate users

- Negative revenue

- Old data not updating

- Schema changes silently breaking dashboards

👉 dbt tests help you **catch these problems early**, automatically.

-------------------

#### **2️⃣ How dbt tests work (internally)**

When you run:


In [ ]:
dbt test

dbt does this:

- Converts tests into **SQL queries**

- Executes them in your **data warehouse**

- Checks the **result count**

    - **✅ 0 rows returned** → test **PASS**

    - **❌ >0 rows returned** → test **FAIL**

> Important rule: **A dbt test fails if it returns rows**

--------------

#### **3️⃣ Types of tests in dbt**

dbt has **two main types** of tests:

| Type               | Defined in | Purpose                          |
| ------------------ | ---------- | -------------------------------- |
| **Generic tests**  | `.yml`     | Column-level & model-level rules |
| **Singular tests** | `.sql`     | Custom business logic            |


---------------

#### **4️⃣ Generic Tests (most commonly used)**

Generic tests are **prebuilt test templates** provided by dbt.

**Built-in generic tests:**

- `not_null`

- `unique`

- `accepted_values`

- `relationships`

These are defined inside **schema.yml** files.

**✅ Example Model**

In [ ]:
-- models/dim_customers.sql
select
  id as customer_id,
  name,
  email,
  country
from {{ ref('raw_customers') }}


--------------

**4.1 `not_null` test**

**Purpose**

Ensures a column **never contains NULL values**

**YAML**

In [ ]:
version: 2

models:
  - name: dim_customers
    columns:
      - name: customer_id
        tests:
          - not_null

**Generated SQL (conceptually)**

In [ ]:
select *
from dim_customers
where customer_id is null

✔ Pass → no NULLs

❌ Fail → NULLs exist


-------------

**4.2 `unique` test**

**Purpose**

Ensures **no duplicate values**

**YAML**

In [ ]:
      - name: customer_id
        tests:
          - unique

**Generated SQL**

In [ ]:
select customer_id, count(*)
from dim_customers
group by customer_id
having count(*) > 1

✔ Pass → no duplicates

❌ Fail → duplicates found

-----------------

**4.3 `accepted_values` test**

**Purpose**

Ensures column values come from a **fixed allowed list**

**YAML**

In [ ]:
      - name: country
        tests:
          - accepted_values:
              values: ['IN', 'US', 'UK']

**Generated SQL**

In [ ]:
select *
from dim_customers
where country not in ('IN','US','UK')

✔ Pass → all valid

❌ Fail → unexpected value found

-------------------

**4.4 `relationships` test (very important 🔥)**

**Purpose**

Validates **foreign key relationships**

**Example**

- `orders.customer_id`

- `customers.customer_id`

**YAML**

In [ ]:
      - name: customer_id
        tests:
          - relationships:
              to: ref('dim_customers')
              field: customer_id

**Generated SQL**

In [ ]:
select *
from fct_orders o
left join dim_customers c
  on o.customer_id = c.customer_id
where c.customer_id is null

✔ Pass → all orders have customers

❌ Fail → orphan records exist

-----------

#### **5️⃣ Singular Tests (custom SQL tests)**

Singular tests are **pure SQL files** placed in the `tests/` folder.

Use them when:

- Built-in tests are **not enough**

- You need **business logic validation**


----------------

**Example 1: Revenue should never be negative**

**File**

In [ ]:
-- tests/no_negative_revenue.sql
select *
from {{ ref('fct_orders') }}
where revenue < 0

✔ Pass → no rows

❌ Fail → negative revenue exists

----------------

**Example 2: Orders should not be in the future**


In [ ]:
select *
from {{ ref('fct_orders') }}
where order_date > current_date

----

**6️⃣ Tests on Sources (raw data checks)**

You can test **raw tables** before transformation.

**Source YAML**

In [ ]:
version: 2

sources:
  - name: airbnb
    tables:
      - name: hosts
        columns:
          - name: id
            tests:
              - not_null
              - unique

💡 This ensures **bad data never enters models**

----------


#### **7️⃣ Severity levels (warn vs error)**

By default → **error**

You can change behavior:

In [ ]:
tests:
  - not_null:
      severity: warn

| Severity | Result                |
| -------- | --------------------- |
| `error`  | dbt run **fails**     |
| `warn`   | dbt run **continues** |

Used when:

- Data issues are acceptable temporarily

- Monitoring rather than blocking


-------------


#### **8️⃣ Running tests**

**Run all tests**

In [ ]:
dbt test

**Run tests for one model**

In [ ]:
dbt test --select dim_customers

**Run only source tests**

In [ ]:
dbt test --select source:*

-------

#### **9️⃣ Where test results are stored?**

dbt creates:

- Test result tables (temporary)

- Logs in `target/`

- Metadata for **lineage & observability**

In dbt Cloud → visible in **UI dashboards**

-------------

#### **🔁 How tests fit into dbt workflow**

In [ ]:
sources → tests → models → tests → dashboards

✔ Catch issues early

✔ Trust analytics

✔ Safe deployments

---------------

#### **🔟 Best Practices (very important)**

✅ Always test:

- Primary keys → `not_null + unique`

- Foreign keys → `relationships`

- Enum columns → `accepted_values`

- ✅ Write **business-rule tests**
- ✅ Test **sources**
- ✅ Use `warn` wisely
- ✅ Run tests in **CI/CD**

-------------------

**What is `dbt test` -x?**

**Short answer**

👉 `dbt test -x` **tells dbt to exclude failing tests instead of failing the entire run.**

It is mainly used when:

- You **know some tests are failing**

- You **don’t want the pipeline to stop**

- You want dbt to **skip / ignore failures temporarily**

-----------

**Break it down clearly**

**Normal behavior (default)**

In [ ]:
dbt test

- If **any test fails ❌**

- dbt exits with **non-zero status**

- CI/CD or job **fails**

This is the **safe & strict mode.**

-------------

**With `-x`**

In [ ]:
dbt test -x

- Tests **still run**

- Failures are **reported**

- **❌ BUT dbt does NOT fail the command**

- Exit code = **success**

So dbt says:

> “I ran the tests, some failed, but I won’t stop the workflow.”

-------------

**What does `-x` literally mean?**

`-x` = **exclude failures from causing failure**

You can think of it as:

In [ ]:
Run tests
↓
If test fails → log it
↓
Do NOT fail the dbt command

-----

#### **Example scenario (very realistic)**

**You run:**

In [ ]:
dbt test

Output:

In [ ]:
FAIL 1 not_null_dim_customers_customer_id

Result:

❌ Pipeline fails

❌ Job stops

------------

**Now with `-x`**

In [ ]:
dbt test -x

Output:

In [ ]:
FAIL 1 not_null_dim_customers_customer_id

Result:

✅ Command succeeds

✅ Pipeline continues

⚠️ You still see the failure